# Q2 — Text-Driven Image Segmentation with SAM 2
This notebook outlines a Colab-friendly pipeline that converts a text prompt into region seeds (via a detector like GroundingDINO/CLIPSeg/GLIP),
then feeds seeds into SAM 2 to get segmentation masks. Some cells include placeholders for model weights — download them when prompted in Colab.


In [ ]:
# Install dependencies (run in Colab)
# You may need to restart the runtime after installing heavy packages.
!pip install -q git+https://github.com/facebookresearch/segment-anything.git
!pip install -q opencv-python-headless
!pip install -q transformers
# GroundingDINO and other detectors may require custom installs — include below as needed (examples commented)
# !git clone https://github.com/IDEA-Research/GroundingDINO.git
# %cd GroundingDINO
# !pip install -q -r requirements.txt
# (Download pretrained weights in Colab and load them in the model)


In [ ]:
# Simple SAM usage example (assuming segment_anything is installed)
from segment_anything import sam_model_registry, SamPredictor
import numpy as np
import cv2
from PIL import Image
import torch

# NOTE: You must download SAM weights (checkpoint) on Colab and provide the path
# For example: sam_vit_h.pth (download link from Segment Anything GitHub / Meta research)
sam_checkpoint = "sam_vit_h.pth"  # <-- download in Colab and set this path
model_type = "vit_h"

try:
    sam = sam_model_registry[model_type](checkpoint=sam_checkpoint)
except Exception as e:
    print("Make sure SAM weights are downloaded in Colab and path is correct.", e)
    raise

predictor = SamPredictor(sam)

# Load image (example)
img_path = "image.jpg"  # upload an example image in Colab or change this path
image = cv2.imread(img_path)
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

predictor.set_image(image)

# Example: if you had box or point prompts from a text-to-region model, you'd pass them here:
# boxes = np.array([[x0, y0, x1, y1]])  # normalized or pixel coords depending on predictor version
# masks, scores, logits = predictor.predict(box=boxes, multimask_output=False)

# For demo, we just pick a center point
h,w = image.shape[:2]
point = np.array([[w//2, h//2]])
label = np.array([1])
masks, scores, logits = predictor.predict(point_coords=point, point_labels=label, multimask_output=False)
mask = masks[0]

# Overlay mask
from matplotlib import pyplot as plt
plt.figure(figsize=(8,8))
plt.imshow(image)
plt.imshow(mask, alpha=0.5)
plt.axis('off')
plt.show()


## Notes and limitations
- To convert text -> seeds you can use GroundingDINO / GLIP / CLIPSeg. These require their own model weights.
- On Colab: download model weights to the VM (e.g., via `wget` or `gdown`) and point the notebook to those files.
- SAM 2 may be gated or require special weights; this notebook uses the 'segment-anything' API as an example.
- The final mask quality depends heavily on the detector + SAM checkpoint.

